In [1]:
import wiki_tools.data_insert as insert
import wiki_tools.web_queries as queries
import pandas as pd
import numpy as np
import json

In [3]:
file = 'flatfiles/played.csv'
all_cols = ["musician_id","instrument_id"]
date_cols = None

In [4]:
new_df = insert.flatfile_to_df(file, all_cols, date_cols)
print(new_df.dtypes)
new_df

musician_id      str
instrument_id    str
dtype: object


,musician_id,instrument_id
0,Q101314,Q9798
1,Q102003,Q80019
2,Q102003,Q17172850
3,Q103651,Q8350
4,Q103767,Q445275
...,...,...
15512,Q118114459,Q80019
15524,Q118372504,Q11405
15525,Q118372504,Q9798
15536,Q118780183,Q128309


In [5]:
# CHECK IF DATABASE CAN HANDLE LONG NAMES (please)
for i in new_df.columns:
    longest = max(new_df[i].str.len())
    print(longest)
    print(new_df[new_df[i].str.len() == longest])

10
      musician_id instrument_id
11817  Q100243124         Q8350
11818  Q100243124       Q495529
11819  Q100243124       Q382616
11820  Q100249671         Q8338
11821  Q100249671       Q382616
...           ...           ...
15512  Q118114459        Q80019
15524  Q118372504        Q11405
15525  Q118372504         Q9798
15536  Q118780183       Q128309
15545  Q119050056         Q5994

[444 rows x 2 columns]
10
     musician_id instrument_id
4080    Q2347469    Q111824539
4610     Q553921    Q116680092


In [6]:
table = 'temp_played'
insert.df_to_db(new_df, table)

2026-10-14 03:08:47,287 INFO sqlalchemy.engine.Engine SELECT DATABASE()
2026-10-14 03:08:47,288 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-10-14 03:08:47,288 INFO sqlalchemy.engine.Engine SELECT @@sql_mode
2026-10-14 03:08:47,289 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-10-14 03:08:47,289 INFO sqlalchemy.engine.Engine SELECT @@lower_case_table_names
2026-10-14 03:08:47,289 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-10-14 03:08:47,290 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-10-14 03:08:47,290 INFO sqlalchemy.engine.Engine DESCRIBE `ijmdb`.`temp_played`
2026-10-14 03:08:47,291 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-10-14 03:08:47,322 INFO sqlalchemy.engine.Engine INSERT INTO temp_played (musician_id, instrument_id) VALUES (%(musician_id)s, %(instrument_id)s)
2026-10-14 03:08:47,322 INFO sqlalchemy.engine.Engine [generated in 0.02718s] [{'musician_id': 'Q101314', 'instrument_id': 'Q9798'}, {'musician_id': 'Q102003', 'instrument_id': 'Q80019'}, {'music

Individual inserts here:

In [ ]:
entities = queries.get_page_from_id('') 
if entities:
    entity_0 = entities[0]
    item = queries.entity_to_item(entity_0, 'musician')
    json.dumps(item)

In [ ]:
if item:
    # Handle dates as we did before
    new_dob = item['dob'].split("-")
    item['year_of_birth'] = new_dob[0]
    item['month_of_birth'] = new_dob[1]
    item['day_of_birth'] = new_dob[2]
    new_dod = item['dod'].split("-")
    item['year_of_death'] = new_dod[0]
    item['month_of_death'] = new_dod[1]
    item['day_of_death'] = new_dod[2]
    rows = []
    inst = item['inst']
    new_item = item.copy()
    del new_item['dob']
    del new_item['dod']
    del new_item['inst']
    for id in inst:
        rows.append((item['entity_id'], id))

In [ ]:
ind_df = pd.DataFrame(data=new_item, index=np.arange(1))
ind_df

In [ ]:
insert.df_to_db(ind_df, "musicians")

In [ ]:
inst_df = pd.DataFrame(data=rows)
inst_df = inst_df.rename(columns={0: 'musician_id', 1: 'instrument_id'})
inst_df

In [ ]:
# insert.df_to_db(inst_df, "played")